<a href="https://colab.research.google.com/github/swapnasaina274/malicious-mobile-app-detection/blob/main/Agentic_AI_Integration_Finished_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agentic AI Integration — Android Malicious Threat Detection (Finished)

**Project:** Android Malicious Threat Detection using Agentic AI
**Team repo:** github.com/swapnasaina274/malicious-mobile-app-detection

This notebook is the finished Agentic AI integration part of the project. It connects all the trained models together through one **agent coordinator**, and uses **Qwen (Qwen/Qwen2.5-3B-Instruct)** as the reporting agent that explains the result in simple words.

## How the system works (simple words)
1. An APK file comes in.
2. Two things are extracted from it: a **byteplot image** (the app's code drawn as a picture) and an **API-call sequence** (the list of functions the app calls, in order).
3. The agent coordinator calls each detector as a **tool**:
   - **ResNet50** reads the byteplot image.
   - **CNN-BiLSTM-Attention** reads the API-call sequence.
   - **LightGBM** reads a 470-number feature vector (this vector comes from *running* an app, so for a raw APK upload it honestly reports itself unavailable — it is demonstrated separately in Step 8).
4. The coordinator combines the answers by **majority vote**. If the models disagree, the disagreement is shown, never hidden.
5. **Qwen** receives the detector outputs and writes the final report + answers questions in a chat. Qwen never invents a result — it only explains what the detectors said.

## How to run (Google Colab)
1. **Runtime → Change runtime type → GPU (T4) → Save**
2. **Runtime → Run all**
3. When asked, allow Google Drive access — the trained model files are in the Drive folder **Malicious project work**.

Everything below runs top to bottom. Each step has a short explanation before the code.

## Step 1 — Setup

Installs the extra libraries Colab does not ship with, and checks whether a GPU is available. Qwen and the CNN models run much faster on GPU.

In [4]:
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running in Google Colab:", IN_COLAB)

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "androguard", "gradio", "lightgbm", "joblib",
                    "transformers", "accelerate"], check=False)

import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU — in Colab use Runtime > Change runtime type > GPU, then Run all again.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

Running in Google Colab: True
PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


## Step 2 — Connect Google Drive and find the model files

The trained model files are too large for GitHub, so the team keeps them in the Google Drive folder **Malicious project work**. This step mounts Drive and checks that every needed file is present.

In [5]:
import os
from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/Malicious project work")
else:
    PROJECT_ROOT = Path(os.environ.get("AMD_PROJECT_ROOT", "."))

if not PROJECT_ROOT.exists():
    print("!! Project folder NOT found:", PROJECT_ROOT)
    print("!! This folder is SHARED with you (owner: the team), so Colab cannot")
    print("!! see it until you add it to your own Drive:")
    print("!!   Google Drive > Shared with me > right-click 'Malicious project work'")
    print("!!   > Add shortcut to Drive > My Drive. Then run this cell again.")

# Where each file actually lives on the team Drive (checked Nov 2026):
# - the .pkl files and the sample APK sit in the project folder itself
# - resnet50_best.pth sits in  Byteplot images/model_outputs/
# - cnn_bilstm_attention_best.pth sits in  Sequence/experiments/cnn_bilstm_attention/
SEARCH_DIRS = [
    PROJECT_ROOT,
    PROJECT_ROOT / "Agentic AI",
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "Byteplot images" / "model_outputs",
    PROJECT_ROOT / "Sequence" / "experiments" / "cnn_bilstm_attention",
]

def find_file(name):
    for d in SEARCH_DIRS:
        p = d / name
        if p.exists():
            return p
    # Fallback: search the whole project tree (slower on Drive, but sure).
    if PROJECT_ROOT.exists():
        for p in PROJECT_ROOT.rglob(name):
            return p
    return None

NEEDED_FILES = [
    "resnet50_best.pth",               # ResNet50 byteplot model
    "cnn_bilstm_attention_best.pth",   # CNN-BiLSTM-Attention sequence model (vocab inside)
    "champion_lightgbm_model.pkl",     # LightGBM tabular model
    "scaler.pkl", "class_mapping.pkl", "feature_names.pkl",
]
print("Project root:", PROJECT_ROOT)
for name in NEEDED_FILES:
    found = find_file(name)
    print(("FOUND  " if found else "MISSING"), name, ("-> " + str(found)) if found else "")

SAMPLE_APK = Path(os.environ.get("AMD_SAMPLE_APK", PROJECT_ROOT / "com.iris.gallery_16.apk"))
print("Sample APK:", SAMPLE_APK, "| exists:", SAMPLE_APK.exists())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project root: /content/drive/MyDrive/Malicious project work
FOUND   resnet50_best.pth -> /content/drive/MyDrive/Malicious project work/Byteplot images/model_outputs/resnet50_best.pth
FOUND   cnn_bilstm_attention_best.pth -> /content/drive/MyDrive/Malicious project work/Sequence/experiments/cnn_bilstm_attention/cnn_bilstm_attention_best.pth
FOUND   champion_lightgbm_model.pkl -> /content/drive/MyDrive/Malicious project work/champion_lightgbm_model.pkl
FOUND   scaler.pkl -> /content/drive/MyDrive/Malicious project work/scaler.pkl
FOUND   class_mapping.pkl -> /content/drive/MyDrive/Malicious project work/class_mapping.pkl
FOUND   feature_names.pkl -> /content/drive/MyDrive/Malicious project work/feature_names.pkl
Sample APK: /content/drive/MyDrive/Malicious project work/com.iris.gallery_16.apk | exists: True


## Step 3 — The CNN-BiLSTM-Attention network definition

The checkpoint file only stores the trained weights, so the network class must be defined exactly as it was during training before the weights can be loaded. (ResNet50 comes from torchvision, so it needs no custom class.)

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

PAD_IDX = 0

class CNNBiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, cnn_channels=128, hidden_dim=128,
                 num_classes=5, dropout=0.4, padding_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        self.conv = nn.Conv1d(embedding_dim, cnn_channels, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(
            input_size=cnn_channels, hidden_size=hidden_dim, num_layers=1,
            batch_first=True, bidirectional=True,
        )
        self.attention = nn.Linear(hidden_dim * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, input_ids, lengths):
        x = self.embedding(input_ids)
        x = x.transpose(1, 2)
        x = F.relu(self.conv(x))
        x = x.transpose(1, 2)
        lengths_cpu = lengths.detach().cpu().clamp(min=1, max=input_ids.size(1))
        packed = pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        output, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=input_ids.size(1))
        scores = self.attention(output).squeeze(-1)
        positions = torch.arange(input_ids.size(1), device=input_ids.device).unsqueeze(0)
        mask = positions >= lengths.to(input_ids.device).unsqueeze(1)
        scores = scores.masked_fill(mask, -1e9)
        attention_weights = torch.softmax(scores, dim=1)
        context = torch.sum(output * attention_weights.unsqueeze(-1), dim=1)
        return self.classifier(self.dropout(context))

## Step 4 — Load all the trained models

Each model is loaded once and kept in memory. If one file is missing, that model is marked unavailable and the rest of the system still works — the status printout at the end shows exactly what loaded.

In [7]:
import numpy as np

VISION_CLASSES = ["adware", "banking", "benign", "riskware", "sms"]
TABULAR_TO_CANONICAL = {
    "Adware": "adware", "Banking": "banking", "SMS Malware": "sms",
    "Riskware": "riskware", "Benign": "benign",
}
LIGHTGBM_UNAVAILABLE_REASON = (
    "LightGBM was trained on 470 syscall/binder frequency features that are "
    "collected by running an app (dynamic analysis, CICMalDroid). A raw APK "
    "file cannot produce that vector, so LightGBM runs only when a feature "
    "vector is supplied (see Step 8)."
)

_state = {}
_errors = {}

def _ok(model, class_names, probs):
    probs = [float(p) for p in probs]
    idx = int(np.argmax(probs))
    return {
        "model": model, "status": "ok",
        "predicted_class": class_names[idx],
        "confidence": round(probs[idx], 4),
        "probabilities": {name: round(p, 4) for name, p in zip(class_names, probs)},
    }

def _fail(model, exc):
    return {"model": model, "status": "error", "reason": f"{type(exc).__name__}: {exc}"}

def load_all():
    import joblib
    from torchvision import models

    # ---- LightGBM (tabular) ----
    try:
        _state["scaler"] = joblib.load(find_file("scaler.pkl"))
        _state["lightgbm"] = joblib.load(find_file("champion_lightgbm_model.pkl"))
        _state["class_mapping"] = joblib.load(find_file("class_mapping.pkl"))
        _state["feature_names"] = joblib.load(find_file("feature_names.pkl"))
    except Exception as e:
        _errors["lightgbm"] = f"{type(e).__name__}: {e}"

    # ---- ResNet50 (byteplot) ----
    try:
        ck = torch.load(find_file("resnet50_best.pth"), map_location=DEVICE, weights_only=False)
        sd = ck["model_state_dict"] if isinstance(ck, dict) else ck
        class_names = list(ck.get("class_names", VISION_CLASSES)) if isinstance(ck, dict) else VISION_CLASSES
        net = models.resnet50(weights=None)
        net.fc = nn.Linear(net.fc.in_features, len(class_names))
        net.load_state_dict({k.removeprefix("module."): v for k, v in sd.items()})
        net.eval().to(DEVICE)
        _state["resnet50"] = (net, class_names)
    except Exception as e:
        _errors["resnet50"] = f"{type(e).__name__}: {e}"

    # ---- CNN-BiLSTM-Attention (API sequence) ----
    try:
        ck = torch.load(find_file("cnn_bilstm_attention_best.pth"), map_location=DEVICE, weights_only=False)
        sd = ck["model_state_dict"]
        vocab = ck["vocab"]
        class_names = [str(c) for c in ck["class_names"]]
        emb_dim = sd["embedding.weight"].shape[1]
        cnn_channels = sd["conv.weight"].shape[0]
        hidden_dim = sd["classifier.weight"].shape[1] // 2
        net = CNNBiLSTMAttention(
            vocab_size=len(vocab), embedding_dim=emb_dim,
            cnn_channels=cnn_channels, hidden_dim=hidden_dim,
            num_classes=len(class_names),
        )
        net.load_state_dict(sd)
        net.eval().to(DEVICE)
        _state["bilstm"] = (net, class_names)
        _state["bilstm_vocab"] = vocab
        _state["bilstm_max_len"] = int(ck.get("max_len", 1000))
    except Exception as e:
        _errors["bilstm"] = f"{type(e).__name__}: {e}"

load_all()
print("Model status:")
for key in ["lightgbm", "resnet50", "bilstm"]:
    print(" ", key, "->", "loaded" if key in _state else _errors.get(key, "not loaded"))

Model status:
  lightgbm -> loaded
  resnet50 -> loaded
  bilstm -> loaded


## Step 5 — Turn the APK's code into a byteplot image

A byteplot draws the raw bytes of the app's `classes.dex` file as an RGB image. Malware families produce visibly different patterns, which is what ResNet50 learned to read. This generator is the team's own — it reproduces the exact channel layout used in training.

In [8]:
import numpy as np
from PIL import Image

def generate_precise_byteplot(dex_bytes, target_size=(256, 256)):
    # Red: small header/metadata footprint at the top.
    # Blue: initial segment of bytecode, rows 0-51.
    # Green: remainder of bytecode, rows 48-255.
    H, W = target_size
    R_channel = np.zeros((H, W), dtype=np.uint8)
    G_channel = np.zeros((H, W), dtype=np.uint8)
    B_channel = np.zeros((H, W), dtype=np.uint8)

    data = np.frombuffer(dex_bytes, dtype=np.uint8)
    total_bytes = len(data)

    header_len = min(40, total_bytes)
    R_channel[0, :header_len] = data[:header_len]

    blue_rows = 52
    blue_pixel_count = blue_rows * W
    blue_bytes_limit = min(blue_pixel_count, total_bytes)
    blue_data = data[:blue_bytes_limit]
    B_channel[:blue_rows, :].flat[:len(blue_data)] = blue_data

    green_start_offset = max(0, blue_bytes_limit - (4 * W))
    green_data = data[green_start_offset:]
    green_rows_avail = H - 48
    green_pixel_count = green_rows_avail * W
    if len(green_data) > green_pixel_count:
        green_data = green_data[:green_pixel_count]
    G_channel[48:, :].flat[:len(green_data)] = green_data

    rgb_image = np.stack([R_channel, G_channel, B_channel], axis=-1)
    return Image.fromarray(rgb_image, mode="RGB")

## Step 6 — Extract everything the detectors need from an APK

From one APK file this step produces:
1. the **byteplot image** (for ResNet50),
2. the **API-call token sequence** (for CNN-BiLSTM) — the names of the methods the app invokes, in DEX order, mapped to numbers through the vocabulary stored in the BiLSTM checkpoint,
3. **static manifest information** — package name, permissions, and simple danger flags (for the report).

In [9]:
import re
import zipfile
from pathlib import Path

try:
    from loguru import logger as _loguru
    _loguru.disable("androguard")
except Exception:
    pass

_INVOKE_RE = re.compile(r"->([A-Za-z0-9_$<>]+)\(")

DANGEROUS_PERMISSION_FLAGS = {
    "android.permission.MANAGE_EXTERNAL_STORAGE": "Requests broad external-storage access",
    "android.permission.REQUEST_INSTALL_PACKAGES": "Can request installation of other packages",
    "android.permission.SYSTEM_ALERT_WINDOW": "Can draw overlays on top of other apps",
    "android.permission.SEND_SMS": "Can send SMS messages",
    "android.permission.READ_SMS": "Can read SMS messages",
    "android.permission.BIND_ACCESSIBILITY_SERVICE": "Requests an accessibility service binding",
    "android.permission.REQUEST_DELETE_PACKAGES": "Can request deletion of packages",
}

def read_dex_files(apk_path):
    # Return {name: bytes} for every classes*.dex entry in the APK.
    out = {}
    with zipfile.ZipFile(apk_path) as zf:
        for name in zf.namelist():
            base = Path(name).name
            if base.startswith("classes") and base.endswith(".dex"):
                out[base] = zf.read(name)
    return out

def byteplot_image(apk_path):
    dexes = read_dex_files(apk_path)
    if "classes.dex" not in dexes:
        raise ValueError("No classes.dex found in APK")
    return generate_precise_byteplot(dexes["classes.dex"]), len(dexes["classes.dex"])

def api_call_tokens(apk_path):
    # Ordered API/method-name tokens from invoke instructions in all DEX files.
    from androguard.core.dex import DEX
    dexes = read_dex_files(apk_path)
    tokens = []
    for name in sorted(dexes):
        try:
            dex = DEX(dexes[name])
        except Exception:
            continue
        for cls in dex.get_classes():
            for method in cls.get_methods():
                code = method.get_code()
                if code is None:
                    continue
                for ins in code.get_bc().get_instructions():
                    if ins.get_name().startswith("invoke"):
                        match = _INVOKE_RE.search(ins.get_output())
                        if match:
                            tokens.append(match.group(1))
    return tokens

def tokens_to_ids(tokens, vocab, max_len=1000):
    # Map tokens through the checkpoint vocabulary (PAD=0, UNK=1).
    unk = vocab.get("<UNK>", 1)
    ids = [vocab.get(tok, unk) for tok in tokens[:max_len]]
    length = max(1, len(ids))
    padded = ids + [0] * (max_len - len(ids))
    return padded, length

def static_info(apk_path):
    # Manifest-level static analysis.
    from androguard.core.apk import APK
    app = APK(str(apk_path))
    permissions = sorted(app.get_permissions())
    flags = [flag for perm, flag in DANGEROUS_PERMISSION_FLAGS.items() if perm in permissions]
    return {
        "package_name": app.get_package(),
        "app_name": app.get_app_name(),
        "version_name": app.get_androidversion_name(),
        "version_code": app.get_androidversion_code(),
        "min_sdk": app.get_min_sdk_version(),
        "target_sdk": app.get_target_sdk_version(),
        "permissions": permissions,
        "permission_count": len(permissions),
        "activity_count": len(app.get_activities()),
        "service_count": len(app.get_services()),
        "receiver_count": len(app.get_receivers()),
        "provider_count": len(app.get_providers()),
        "review_flags": flags,
        "static_analysis_only": True,
    }

## Step 7 — The agent: detector tools + coordinator

This is the "agentic" part. Each trained model is wrapped as a **tool** with the same input/output shape. The **coordinator** (`run_detectors`) calls whichever tools have their inputs available, then `combine` decides the final class by **majority vote** (a tie is broken by mean confidence). A tool whose input was not supplied reports itself *unavailable with a reason* instead of guessing.

In [10]:
# ---------- single-model prediction functions ----------

def _eval_transform():
    from torchvision import transforms
    return transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def predict_vision(byteplot_pil_image):
    # ResNet50 on one byteplot image. (MobileNetV3 was trained for
    # comparison; ResNet50 scored higher, 0.904 vs 0.883, so only
    # ResNet50 serves predictions in the final system.)
    results = []
    tensor = _eval_transform()(byteplot_pil_image.convert("RGB")).unsqueeze(0).to(DEVICE)
    if "resnet50" not in _state:
        return [{"model": "ResNet50", "status": "unavailable",
                 "reason": _errors.get("resnet50", "model not loaded")}]
    net, class_names = _state["resnet50"]
    try:
        with torch.inference_mode():
            probs = torch.softmax(net(tensor), dim=1)[0].cpu().numpy()
        results.append(_ok("ResNet50", class_names, probs))
    except Exception as e:
        results.append(_fail("ResNet50", e))
    return results

def predict_sequence(padded_ids, length, token_count):
    label = "CNN-BiLSTM"
    if "bilstm" not in _state:
        return {"model": label, "status": "unavailable",
                "reason": _errors.get("bilstm", "model not loaded")}
    net, class_names = _state["bilstm"]
    try:
        ids = torch.tensor([padded_ids], dtype=torch.long).to(DEVICE)
        lengths = torch.tensor([length], dtype=torch.long)
        with torch.inference_mode():
            probs = torch.softmax(net(ids, lengths), dim=1)[0].cpu().numpy()
        result = _ok(label, class_names, probs)
        result["tokens_used"] = token_count
        return result
    except Exception as e:
        return _fail(label, e)

def predict_vector(values):
    # LightGBM on a 470-feature vector: a list of 470 numbers, or a
    # {feature_name: value} dict (missing features default to 0).
    import pandas as pd
    label = "LightGBM"
    if "lightgbm" not in _state:
        return {"model": label, "status": "unavailable",
                "reason": _errors.get("lightgbm", "model not loaded")}
    fnames = list(_state["feature_names"])
    if isinstance(values, dict):
        row = [float(values.get(name, 0.0)) for name in fnames]
    else:
        row = [float(v) for v in values]
        if len(row) != len(fnames):
            return {"model": label, "status": "error",
                    "reason": f"expected {len(fnames)} feature values, got {len(row)}"}
    try:
        x = pd.DataFrame([row], columns=fnames)
        probs = _state["lightgbm"].predict_proba(x)[0]
        cmap = _state["class_mapping"]
        canonical_probs = {}
        for idx, p in enumerate(probs):
            canonical_probs[TABULAR_TO_CANONICAL[cmap[int(idx)]]] = float(p)
        best = max(canonical_probs, key=canonical_probs.get)
        return {
            "model": label, "status": "ok", "predicted_class": best,
            "confidence": round(canonical_probs[best], 4),
            "probabilities": {k: round(v, 4) for k, v in canonical_probs.items()},
        }
    except Exception as e:
        return _fail(label, e)

# ---------- agent tools ----------

def resnet50_tool(image):
    return predict_vision(image)[0]

def bilstm_tool(padded_ids, length, token_count):
    return predict_sequence(padded_ids, length, token_count)

def lightgbm_tool(feature_vector):
    return predict_vector(feature_vector)

AGENT_TOOLS = {"resnet50": resnet50_tool, "bilstm": bilstm_tool, "lightgbm": lightgbm_tool}

# ---------- ensemble (majority vote) ----------

def combine(detector_outputs):
    valid = [d for d in detector_outputs if d.get("status") == "ok"]
    if not valid:
        return {"final_prediction": None, "agreement": "0/0",
                "disagreement_detected": False, "mean_confidence": None}
    votes = {}
    for det in valid:
        votes.setdefault(det["predicted_class"], []).append(det)
    counts = {cls: len(v) for cls, v in votes.items()}
    top_count = max(counts.values())
    leaders = [cls for cls, n in counts.items() if n == top_count]
    if len(leaders) == 1:
        winner = leaders[0]
    else:
        winner = max(leaders, key=lambda cls: sum(d["confidence"] for d in votes[cls]) / len(votes[cls]))
    mean_conf = sum(d["confidence"] for d in votes[winner]) / len(votes[winner])
    return {
        "final_prediction": winner,
        "agreement": f"{counts[winner]}/{len(valid)}",
        "disagreement_detected": len(votes) > 1,
        "mean_confidence": round(mean_conf, 4),
        "votes": {cls: [d["model"] for d in v] for cls, v in votes.items()},
    }

# ---------- the coordinator ----------

def run_detectors(image=None, api_sequence=None, lgbm_features=None):
    # Calls every tool whose input was supplied; a tool without input
    # reports itself unavailable, with the reason, instead of guessing.
    results = []
    if image is not None:
        results.append(resnet50_tool(image))
    else:
        results.append({"model": "ResNet50", "status": "unavailable",
                        "reason": "byteplot image not supplied"})
    if api_sequence is not None:
        padded_ids, length, token_count = api_sequence
        results.append(bilstm_tool(padded_ids, length, token_count))
    else:
        results.append({"model": "CNN-BiLSTM", "status": "unavailable",
                        "reason": "API sequence not supplied"})
    if lgbm_features is not None:
        results.append(lightgbm_tool(lgbm_features))
    else:
        results.append({"model": "LightGBM", "status": "unavailable",
                        "reason": LIGHTGBM_UNAVAILABLE_REASON})
    return results

def analyze_apk(apk_path):
    # Full pipeline for one APK: extract -> run agent tools -> vote.
    apk_path = Path(apk_path)
    image, dex_size = byteplot_image(apk_path)
    tokens = api_call_tokens(apk_path)
    vocab = _state.get("bilstm_vocab")
    max_len = _state.get("bilstm_max_len", 1000)
    api_sequence = None
    if vocab is not None:
        padded, length = tokens_to_ids(tokens, vocab, max_len)
        api_sequence = (padded, length, len(tokens))
    detectors = run_detectors(image=image, api_sequence=api_sequence)
    return {
        "filename": apk_path.name,
        "dex_bytes": dex_size,
        "api_token_count": len(tokens),
        "static": static_info(apk_path),
        "detectors": detectors,
        "ensemble": combine(detectors),
    }

print("Agent ready. Tools:", list(AGENT_TOOLS))

Agent ready. Tools: ['resnet50', 'bilstm', 'lightgbm']


## Step 8 — Load Qwen, the reporting agent

Qwen/Qwen2.5-3B-Instruct is downloaded from Hugging Face the first time this cell runs. Its job is **only** to explain the detector outputs — the classification decision always comes from the detectors and the vote, never from Qwen. If Qwen cannot load, the report step falls back to a plain template summary that is clearly labelled as a template.

In [11]:
import json
import re

QWEN_MODEL_ID = os.environ.get("AMD_QWEN_MODEL", "Qwen/Qwen2.5-3B-Instruct")
_qwen = {}
_qwen_error = None

def _get_qwen():
    global _qwen_error
    if "model" in _qwen:
        return _qwen
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID)
        if DEVICE == "cuda":
            model = AutoModelForCausalLM.from_pretrained(
                QWEN_MODEL_ID, torch_dtype=torch.float16, device_map="auto")
        else:
            model = AutoModelForCausalLM.from_pretrained(
                QWEN_MODEL_ID, torch_dtype=torch.float32)
        model.eval()
        _qwen["tokenizer"] = tokenizer
        _qwen["model"] = model
        print("Qwen loaded:", QWEN_MODEL_ID, "on", DEVICE)
    except Exception as e:
        _qwen_error = f"{type(e).__name__}: {e}"
        print("Qwen could not load:", _qwen_error)
        return None
    return _qwen

def _qwen_chat(messages, max_tokens=600, temperature=0.6):
    q = _get_qwen()
    if q is None:
        raise RuntimeError(_qwen_error or "Qwen unavailable")
    tokenizer, model = q["tokenizer"], q["model"]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    gen_kwargs = {"max_new_tokens": max_tokens, "pad_token_id": tokenizer.eos_token_id}
    if temperature and temperature > 0:
        gen_kwargs.update({"do_sample": True, "temperature": temperature, "top_p": 0.9})
    with torch.inference_mode():
        out = model.generate(**inputs, **gen_kwargs)
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

_get_qwen()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded: Qwen/Qwen2.5-3B-Instruct on cuda


{'tokenizer': Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-3B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
 	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
 	151649: AddedToken("<|bo

## Step 9 — Report generation and chat

`generate_report` gives Qwen the detector outputs as JSON and asks for a structured report (verdict, summary, supporting predictions, unavailable models, limitations, explanation). `chat_answer` answers follow-up questions about the last analysis. Both use strict system prompts: *never invent results; if the models disagree, say so.*

In [12]:
SYSTEM_REPORT = (
    "You are the reporting agent inside an Android malware detection system. "
    "You never invent results; you only explain the detector outputs you are given. "
    "If the models disagree, say so plainly and never present a disagreement as a "
    "definitive verdict. Static analysis alone cannot prove an app is malicious."
)
SYSTEM_CHAT = (
    "You are the assistant inside an Android malware detection app built by an MCA "
    "student team. Answer questions about the analysis results provided as context, "
    "and about Android malware concepts in simple, clear language. Never invent "
    "detector results that are not in the context. If the context has no analysis "
    "yet, say the user should upload an APK first."
)
MODEL_LABEL = "Qwen/Qwen2.5-3B-Instruct (transformers)"

def _context_json(analysis):
    slim = {
        "file": analysis.get("filename"),
        "package_name": (analysis.get("static") or {}).get("package_name"),
        "permissions": (analysis.get("static") or {}).get("permissions", []),
        "review_flags": (analysis.get("static") or {}).get("review_flags", []),
        "detectors": analysis.get("detectors", []),
        "ensemble": analysis.get("ensemble", {}),
    }
    return json.dumps(slim, indent=2)

def template_report(analysis, reason):
    ens = analysis.get("ensemble", {})
    detectors = analysis.get("detectors", [])
    supporting = [f"{d['model']}: {d['predicted_class']}" for d in detectors if d.get("status") == "ok"]
    unavailable = [f"{d['model']}: {d.get('reason', 'unavailable')}" for d in detectors if d.get("status") != "ok"]
    verdict = ens.get("final_prediction") or "unknown"
    agreement = ens.get("agreement", "0/0")
    if ens.get("disagreement_detected"):
        explanation = (
            f"The detectors disagree on this app. The ensemble verdict is {verdict} "
            f"by majority vote ({agreement} agreement). Treat this as inconclusive "
            "rather than a definitive classification."
        )
    else:
        explanation = (
            f"The available detectors agree on {verdict} ({agreement} agreement, "
            f"mean confidence {ens.get('mean_confidence')})."
        )
    return {
        "source": "template", "model": None,
        "note": f"AI report unavailable ({reason}); this is a plain summary of the detector outputs.",
        "verdict": verdict, "summary": explanation,
        "supporting_predictions": supporting, "unavailable_models": unavailable,
        "limitations": ("Static analysis only. LightGBM does not run on raw APK uploads, and "
                        "no dynamic behaviour was observed."),
        "explanation": explanation,
    }

def generate_report(analysis):
    if _get_qwen() is None:
        return template_report(analysis, _qwen_error or "Qwen model not loaded")
    prompt = (
        "Detector outputs and static analysis for one uploaded APK (JSON):\n"
        f"{_context_json(analysis)}\n\n"
        "Return ONLY a JSON object with these keys: "
        "verdict (the ensemble final class), "
        "summary (2-3 plain sentences a non-expert can understand), "
        "supporting_predictions (list of 'Model: class' strings for models that ran), "
        "unavailable_models (list of 'Model: reason' strings), "
        "limitations (one or two sentences), "
        "explanation (a short paragraph: the verdict, the agreement level, any "
        "disagreement, and what in the static evidence matters). "
        "No markdown, no text outside the JSON."
    )
    try:
        text = _qwen_chat(
            [{"role": "system", "content": SYSTEM_REPORT},
             {"role": "user", "content": prompt}],
            max_tokens=700, temperature=0.4,
        )
        match = re.search(r"\{.*\}", text, re.DOTALL)
        data = json.loads(match.group(0)) if match else None
        if not data or "verdict" not in data:
            return template_report(analysis, "Qwen returned an unreadable report")
        data["source"] = "qwen"
        data["model"] = MODEL_LABEL
        return data
    except Exception as e:
        return template_report(analysis, f"{type(e).__name__}: {e}")

def chat_answer(question, analysis=None):
    if _get_qwen() is None:
        return {"source": "unavailable",
                "answer": ("The AI assistant could not be loaded "
                           f"({_qwen_error or 'Qwen model not loaded'}), so chat is unavailable. "
                           "The detector results themselves are unaffected.")}
    context = _context_json(analysis) if analysis else "No APK has been analysed yet."
    try:
        answer = _qwen_chat(
            [{"role": "system", "content": SYSTEM_CHAT},
             {"role": "user", "content": f"Analysis context (JSON):\n{context}\n\nQuestion: {question}"}],
            max_tokens=400, temperature=0.7,
        )
        return {"source": "qwen", "model": MODEL_LABEL, "answer": answer}
    except Exception as e:
        return {"source": "unavailable", "answer": f"The AI assistant hit an error: {e}"}

print("Report + chat functions ready.")

Report + chat functions ready.


## Step 10 — Full run on the sample APK

Runs the whole pipeline on the team's sample APK (`com.iris.gallery_16.apk`, from the Drive project folder): extraction, both APK detectors, the vote, the Qwen report, and one sample chat question.

In [13]:
analysis = analyze_apk(SAMPLE_APK)

print("File:", analysis["filename"])
print("Package:", analysis["static"]["package_name"])
print("API tokens extracted:", analysis["api_token_count"])
print()
for d in analysis["detectors"]:
    if d["status"] == "ok":
        print(f"  {d['model']:<12} -> {d['predicted_class']}  (confidence {d['confidence']})")
    else:
        print(f"  {d['model']:<12} -> {d['status']}: {str(d.get('reason', ''))[:90]}")
ens = analysis["ensemble"]
print()
print("FINAL VERDICT:", ens["final_prediction"], "| agreement:", ens["agreement"],
      "| disagreement:", ens["disagreement_detected"])

report = generate_report(analysis)
print()
print("=== AI REPORT (source:", report.get("source"), ") ===")
print(json.dumps(report, indent=2))

print()
print("=== SAMPLE CHAT ===")
qa = chat_answer("Is this app dangerous? What should I do with it?", analysis)
print(qa["answer"])

/tmp/ipykernel_1316/1957427005.py:34: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(rgb_image, mode="RGB")


File: com.iris.gallery_16.apk
Package: com.iris.gallery
API tokens extracted: 149562

  ResNet50     -> riskware  (confidence 0.8915)
  CNN-BiLSTM   -> adware  (confidence 0.8499)
  LightGBM     -> unavailable: LightGBM was trained on 470 syscall/binder frequency features that are collected by runnin

FINAL VERDICT: riskware | agreement: 1/2 | disagreement: True

=== AI REPORT (source: qwen ) ===
{
  "verdict": "riskware",
  "summary": "The APK has broad storage permissions and is predicted to be riskware by multiple models.",
  "supporting_predictions": [
    "Riskware: ResNet50",
    "Adware: CNN-BiLSTM"
  ],
  "unavailable_models": [
    "LightGBM: Feature vector not available from raw APK"
  ],
  "limitations": "Static analysis alone cannot definitively determine if the app is malicious without dynamic analysis.",
  "explanation": "The ensemble model concludes the APK is riskware based on the majority vote from ResNet50 and CNN-BiLSTM, which both predict riskware. The disagreement 

## Step 11 — LightGBM demonstration (feature vector)

LightGBM was trained on 470 syscall/binder frequency features that are recorded while an app *runs* (dynamic analysis in CICMalDroid 2020), so it cannot score a raw APK file. Here it is demonstrated honestly with one real feature vector taken from the team's own dataset (`lightgbm_demo_vector.json` in the project folder, if present).

In [14]:
demo_vector_file = find_file("lightgbm_demo_vector.json")
if demo_vector_file is not None:
    demo_data = json.loads(Path(demo_vector_file).read_text())
    features = demo_data.get("features", demo_data)
    result = lightgbm_tool(features)
    print("LightGBM on the demo feature vector:")
    print(json.dumps(result, indent=2))
else:
    print("lightgbm_demo_vector.json not found in the project folder — skipping the demo.")
    print("The APK pipeline above is unaffected: LightGBM simply stays 'unavailable' for raw APKs.")

lightgbm_demo_vector.json not found in the project folder — skipping the demo.
The APK pipeline above is unaffected: LightGBM simply stays 'unavailable' for raw APKs.


## Step 12 — Gradio app (upload an APK in the browser)

A small web app on top of the same agent: upload any APK and get the verdict, the per-model results, the AI report, and a chat box for questions about that analysis. In Colab this creates a shareable link that stays alive while the notebook keeps running.

In [15]:
import gradio as gr

LAST = {"analysis": None}

def ui_analyze(file_obj):
    if file_obj is None:
        return "No file uploaded.", "", ""
    result = analyze_apk(file_obj.name)
    LAST["analysis"] = result
    ens = result["ensemble"]
    rows = []
    for d in result["detectors"]:
        if d["status"] == "ok":
            rows.append(f"| {d['model']} | {d['predicted_class']} | {d['confidence']} |")
        else:
            rows.append(f"| {d['model']} | {d['status']} | {str(d.get('reason', ''))[:80]} |")
    table = "| Model | Result | Confidence / Reason |\n|---|---|---|\n" + "\n".join(rows)
    rep = generate_report(result)
    verdict = (f"## Verdict: {ens['final_prediction']}  \n"
               f"Agreement: {ens['agreement']} · Mean confidence: {ens['mean_confidence']} · "
               f"Package: {result['static']['package_name']}")
    return verdict, table, json.dumps(rep, indent=2)

def ui_chat(message, history):
    qa = chat_answer(message, LAST["analysis"])
    return qa["answer"]

with gr.Blocks(title="Android Malicious Threat Detection — Agentic AI") as demo:
    gr.Markdown("# Android Malicious Threat Detection using Agentic AI\n"
                "Upload an APK: ResNet50 (byteplot) + CNN-BiLSTM (API sequence) vote, Qwen explains.")
    apk_in = gr.File(label="Upload APK", file_types=[".apk"])
    verdict_out = gr.Markdown()
    table_out = gr.Markdown()
    report_out = gr.Code(label="AI report (JSON)", language="json")
    apk_in.change(ui_analyze, inputs=apk_in, outputs=[verdict_out, table_out, report_out])
    gr.ChatInterface(fn=ui_chat, title="Ask about the analysis")

if IN_COLAB:
    demo.launch(share=True)
else:
    print("Gradio UI built. (It launches automatically in Google Colab.)")

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://47484cf692fd445ce3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
# ============================================================
#  NEW UI — Android Malicious Threat Detection (Agentic AI)
#  Is poore code ko EK hi cell me paste karke run karna hai.
#  Purani UI apne aap band ho jayegi, nayi UI naye link par khulegi.
# ============================================================
import json
import gradio as gr

# Agar purani UI abhi chal rahi hai to pehle use band karo
try:
    demo.close()
    print("Purani UI band kar di.")
except Exception:
    pass

LAST = {"analysis": None}

VERDICT_COLORS = {
    "benign":   ("#15803d", "#dcfce7"),
    "riskware": ("#b45309", "#fef3c7"),
    "adware":   ("#c2410c", "#ffedd5"),
    "banking":  ("#b91c1c", "#fee2e2"),
    "sms":      ("#b91c1c", "#fee2e2"),
}

def ui_analyze(file_obj):
    if file_obj is None:
        return "", "", "", ""
    path = file_obj if isinstance(file_obj, str) else file_obj.name
    result = analyze_apk(path)
    LAST["analysis"] = result
    ens = result["ensemble"]
    st = result["static"]
    verdict = ens["final_prediction"] or "unknown"
    color, bg = VERDICT_COLORS.get(verdict, ("#334155", "#e2e8f0"))

    verdict_html = f"""
    <div style="background:{bg};border:2px solid {color};border-radius:14px;padding:18px 22px;">
      <div style="font-size:13px;letter-spacing:2px;color:{color};font-weight:700;">FINAL VERDICT</div>
      <div style="font-size:40px;font-weight:800;color:{color};text-transform:capitalize;line-height:1.15;">{verdict}</div>
      <div style="font-size:15px;color:#1e293b;margin-top:4px;">
        Agreement: <b>{ens['agreement']}</b> &nbsp;·&nbsp; Mean confidence: <b>{ens['mean_confidence']}</b>
        &nbsp;·&nbsp; Disagreement: <b>{'Yes' if ens['disagreement_detected'] else 'No'}</b>
      </div>
      <div style="font-size:14px;color:#475569;margin-top:6px;">
        {st.get('app_name') or ''} &nbsp;·&nbsp; {st.get('package_name')} &nbsp;·&nbsp;
        {st.get('permission_count')} permissions &nbsp;·&nbsp; {result['api_token_count']:,} API calls analysed
      </div>
    </div>"""

    rows = ""
    for d in result["detectors"]:
        if d["status"] == "ok":
            c, b = VERDICT_COLORS.get(d["predicted_class"], ("#334155", "#e2e8f0"))
            rows += (f"<tr><td style='padding:10px 14px;font-weight:600;'>{d['model']}</td>"
                     f"<td style='padding:10px 14px;'><span style='background:{b};color:{c};"
                     f"padding:3px 12px;border-radius:999px;font-weight:700;text-transform:capitalize;'>{d['predicted_class']}</span></td>"
                     f"<td style='padding:10px 14px;'>{d['confidence']}</td></tr>")
        else:
            rows += (f"<tr><td style='padding:10px 14px;font-weight:600;'>{d['model']}</td>"
                     f"<td style='padding:10px 14px;color:#b45309;'>unavailable</td>"
                     f"<td style='padding:10px 14px;color:#64748b;font-size:13px;'>{str(d.get('reason', ''))[:110]}</td></tr>")
    table_html = f"""
    <table style="width:100%;border-collapse:collapse;background:#ffffff;border-radius:12px;overflow:hidden;">
      <tr style="background:#0f3557;color:#ffffff;text-align:left;">
        <th style="padding:10px 14px;">Model</th><th style="padding:10px 14px;">Prediction</th>
        <th style="padding:10px 14px;">Confidence / Reason</th>
      </tr>{rows}
    </table>"""

    rep = generate_report(result)
    src_note = "Qwen AI report" if rep.get("source") == "qwen" else "Plain summary (AI unavailable)"
    report_md = f"""### {src_note}
**Summary:** {rep.get('summary', '')}

**Explanation:** {rep.get('explanation', '')}

**Supporting predictions:** {', '.join(rep.get('supporting_predictions', [])) or '—'}

**Limitations:** {rep.get('limitations', '')}"""
    return verdict_html, table_html, report_md, json.dumps(rep, indent=2)

def ui_chat(message, history):
    qa = chat_answer(message, LAST["analysis"])
    return qa["answer"]

CUSTOM_CSS = """
.gradio-container {
  background: linear-gradient(135deg, #0b2545 0%, #0e4d64 55%, #0f766e 100%) !important;
  max-width: 1080px !important; margin: 0 auto !important;
  font-family: 'Segoe UI', system-ui, sans-serif !important;
}
.main-card {
  background: rgba(255,255,255,0.97); border-radius: 16px;
  padding: 20px 22px; box-shadow: 0 10px 30px rgba(0,0,0,0.35);
}
footer { visibility: hidden; }
"""

HERO_HTML = """
<div style="text-align:center;padding:30px 10px 8px 10px;">
  <div style="font-size:42px;">🛡️</div>
  <h1 style="color:#ffffff;font-size:34px;font-weight:800;margin:6px 0 4px 0;">
    Android Malicious Threat Detection</h1>
  <div style="color:#a7f3ea;font-size:17px;font-weight:600;letter-spacing:1px;">
    using Agentic AI</div>
  <p style="color:#dbeafe;font-size:14.5px;max-width:720px;margin:10px auto 0 auto;">
    APK upload karo — <b>ResNet50</b> (byteplot) aur <b>CNN-BiLSTM</b> (API sequence) vote karenge,
    aur <b>Qwen AI</b> report likh kar samjhayega.</p>
</div>"""

with gr.Blocks(title="Android Malicious Threat Detection — Agentic AI",
               theme=gr.themes.Soft(primary_hue="teal", secondary_hue="blue"),
               css=CUSTOM_CSS) as app:
    gr.HTML(HERO_HTML)
    with gr.Column(elem_classes="main-card"):
        gr.Markdown("### 📤 Step 1 — APK upload karo")
        apk_in = gr.File(label="APK file (.apk)", file_types=[".apk"])
        gr.Markdown("### 🔍 Step 2 — Result")
        verdict_out = gr.HTML()
        table_out = gr.HTML()
        gr.Markdown("### 🤖 Step 3 — AI Report")
        report_out = gr.Markdown()
        with gr.Accordion("Poora report JSON me dekho", open=False):
            report_json = gr.Code(label="AI report (JSON)", language="json")
        apk_in.change(ui_analyze, inputs=apk_in,
                      outputs=[verdict_out, table_out, report_out, report_json])
    with gr.Column(elem_classes="main-card"):
        gr.ChatInterface(fn=ui_chat,
                         title="💬 Analysis ke baare me kuch bhi poochho",
                         examples=["Is this app dangerous?",
                                   "Dono models alag-alag kyun bol rahe hain?",
                                   "Riskware ka matlab kya hota hai?"])
    gr.HTML("<p style='text-align:center;color:#dbeafe;font-size:12.5px;padding:6px;'>"
            "MCA Major Project · ResNet50 + CNN-BiLSTM-Attention + LightGBM · "
            "Reporting agent: Qwen2.5-3B-Instruct · Static analysis only</p>")

app.launch(share=True)


Closing server running on port: 7860
Purani UI band kar di.


/tmp/ipykernel_1316/366648499.py:112: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Android Malicious Threat Detection — Agentic AI",


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed812c6966b482dbc3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
# ============================================================
#  NEW UI — Android Malicious Threat Detection (Agentic AI)
#  Paste this entire code into ONE cell and run it.
#  The old UI will close automatically and the new UI will open on a new link.
# ============================================================
import json
import gradio as gr

# If the old UI is still running, close it first
try:
    demo.close()
    print("Previous UI closed.")
except Exception:
    pass

LAST = {"analysis": None}

VERDICT_COLORS = {
    "benign":   ("#15803d", "#dcfce7"),
    "riskware": ("#b45309", "#fef3c7"),
    "adware":   ("#c2410c", "#ffedd5"),
    "banking":  ("#b91c1c", "#fee2e2"),
    "sms":      ("#b91c1c", "#fee2e2"),
}

def ui_analyze(file_obj):
    if file_obj is None:
        return "", "", "", ""
    path = file_obj if isinstance(file_obj, str) else file_obj.name
    result = analyze_apk(path)
    LAST["analysis"] = result
    ens = result["ensemble"]
    st = result["static"]
    verdict = ens["final_prediction"] or "unknown"
    color, bg = VERDICT_COLORS.get(verdict, ("#334155", "#e2e8f0"))

    verdict_html = f"""
    <div style="background:{bg};border:2px solid {color};border-radius:14px;padding:18px 22px;">
      <div style="font-size:13px;letter-spacing:2px;color:{color};font-weight:700;">FINAL VERDICT</div>
      <div style="font-size:40px;font-weight:800;color:{color};text-transform:capitalize;line-height:1.15;">{verdict}</div>
      <div style="font-size:15px;color:#1e293b;margin-top:4px;">
        Agreement: <b>{ens['agreement']}</b> &nbsp;·&nbsp; Mean confidence: <b>{ens['mean_confidence']}</b>
        &nbsp;·&nbsp; Disagreement: <b>{'Yes' if ens['disagreement_detected'] else 'No'}</b>
      </div>
      <div style="font-size:14px;color:#475569;margin-top:6px;">
        {st.get('app_name') or ''} &nbsp;·&nbsp; {st.get('package_name')} &nbsp;·&nbsp;
        {st.get('permission_count')} permissions &nbsp;·&nbsp; {result['api_token_count']:,} API calls analysed
      </div>
    </div>"""

    rows = ""
    for d in result["detectors"]:
        if d["status"] == "ok":
            c, b = VERDICT_COLORS.get(d["predicted_class"], ("#334155", "#e2e8f0"))
            rows += (f"<tr><td style='padding:10px 14px;font-weight:600;'>{d['model']}</td>"
                     f"<td style='padding:10px 14px;'><span style='background:{b};color:{c};"
                     f"padding:3px 12px;border-radius:999px;font-weight:700;text-transform:capitalize;'>{d['predicted_class']}</span></td>"
                     f"<td style='padding:10px 14px;'>{d['confidence']}</td></tr>")
        else:
            rows += (f"<tr><td style='padding:10px 14px;font-weight:600;'>{d['model']}</td>"
                     f"<td style='padding:10px 14px;color:#b45309;'>unavailable</td>"
                     f"<td style='padding:10px 14px;color:#64748b;font-size:13px;'>{str(d.get('reason', ''))[:110]}</td></tr>")
    table_html = f"""
    <table style="width:100%;border-collapse:collapse;background:#ffffff;border-radius:12px;overflow:hidden;">
      <tr style="background:#0f3557;color:#ffffff;text-align:left;">
        <th style="padding:10px 14px;">Model</th><th style="padding:10px 14px;">Prediction</th>
        <th style="padding:10px 14px;">Confidence / Reason</th>
      </tr>{rows}
    </table>"""

    rep = generate_report(result)
    src_note = "Qwen AI report" if rep.get("source") == "qwen" else "Plain summary (AI unavailable)"
    report_md = f"""### {src_note}
**Summary:** {rep.get('summary', '')}

**Explanation:** {rep.get('explanation', '')}

**Supporting predictions:** {', '.join(rep.get('supporting_predictions', [])) or '—'}

**Limitations:** {rep.get('limitations', '')}"""
    return verdict_html, table_html, report_md, json.dumps(rep, indent=2)

def ui_chat(message, history):
    qa = chat_answer(message, LAST["analysis"])
    return qa["answer"]

CUSTOM_CSS = """
.gradio-container {
  background: linear-gradient(135deg, #0b2545 0%, #0e4d64 55%, #0f766e 100%) !important;
  max-width: 1080px !important; margin: 0 auto !important;
  font-family: 'Segoe UI', system-ui, sans-serif !important;
}
.main-card {
  background: rgba(255,255,255,0.97); border-radius: 16px;
  padding: 20px 22px; box-shadow: 0 10px 30px rgba(0,0,0,0.35);
}
footer { visibility: hidden; }
"""

HERO_HTML = """
<div style="text-align:center;padding:30px 10px 8px 10px;">
  <div style="font-size:42px;">🛡️</div>
  <h1 style="color:#ffffff;font-size:34px;font-weight:800;margin:6px 0 4px 0;">
    Android Malicious Threat Detection</h1>
  <div style="color:#a7f3ea;font-size:17px;font-weight:600;letter-spacing:1px;">
    using Agentic AI</div>
  <p style="color:#dbeafe;font-size:14.5px;max-width:720px;margin:10px auto 0 auto;">
    Upload an APK — <b>ResNet50</b> (byteplot) and <b>CNN-BiLSTM</b> (API sequence) vote on it,
    and <b>Qwen AI</b> writes and explains the report.</p>
</div>"""

with gr.Blocks(title="Android Malicious Threat Detection — Agentic AI",
               theme=gr.themes.Soft(primary_hue="teal", secondary_hue="blue"),
               css=CUSTOM_CSS) as app:
    gr.HTML(HERO_HTML)
    with gr.Column(elem_classes="main-card"):
        gr.Markdown("### 📤 Step 1 — Upload the APK")
        apk_in = gr.File(label="APK file (.apk)", file_types=[".apk"])
        gr.Markdown("### 🔍 Step 2 — Result")
        verdict_out = gr.HTML()
        table_out = gr.HTML()
        gr.Markdown("### 🤖 Step 3 — AI Report")
        report_out = gr.Markdown()
        with gr.Accordion("View the full report as JSON", open=False):
            report_json = gr.Code(label="AI report (JSON)", language="json")
        apk_in.change(ui_analyze, inputs=apk_in,
                      outputs=[verdict_out, table_out, report_out, report_json])
    with gr.Column(elem_classes="main-card"):
        gr.ChatInterface(fn=ui_chat,
                         title="💬 Ask anything about the analysis",
                         examples=["Is this app dangerous?",
                                   "Why did the two models give different answers?",
                                   "What does riskware mean?"])
    gr.HTML("<p style='text-align:center;color:#dbeafe;font-size:12.5px;padding:6px;'>"
            "MCA Major Project · ResNet50 + CNN-BiLSTM-Attention + LightGBM · "
            "Reporting agent: Qwen2.5-3B-Instruct · Static analysis only</p>")

app.launch(share=True)


Previous UI closed.


/tmp/ipykernel_1316/1912213435.py:112: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Android Malicious Threat Detection — Agentic AI",


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bab59573fce011cd86.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
